# Section 5 (Bayesian Inference) — Analysis Summary & Writing Plan

Companion notes for `08_bayesian_spectral_constraints_v2.ipynb`. The notebook performs nested sampling on the IP model using a unified likelihood combining the new tight $\dot P$ measurement with Salcedo+2024's spectral observables ($L_X$, $kT_{\max}$). This document records (a) my assessment of the methodology, (b) the key scientific finding and recommended framing, and (c) things to keep in mind while drafting the LaTeX section.

---

## 1. Headline & framing

**Section role**: methods demonstration — combining timing and spectral observables in a unified Bayesian forward model — applied to CJ2056 as a worked example.

**Headline finding**: the new $\dot P$ measurement, despite being ~1000× tighter than Salcedo+2024's upper limit, does **not** tighten parameter posteriors beyond what the spectral data alone already provide. The original hypothesis (tight $\dot P$ → tighter $B$ via the magnetic-torque term) is empirically disconfirmed.

**Why this is the right framing (not a failure)**:
- It diagnoses CJ2056 as sitting close to spin equilibrium ($r_m \approx r_{\rm co}$), where torque balance forces $|\dot P|$ to be small *for any* spectrally-allowed parameter combination, and $\dot P \approx 0$ is consistent with everything.
- It tells future workers *when* pulsar timing of CVs adds parameter leverage and when it doesn't. Tight $\dot P$ is informative only when the predicted $|\dot P|$ from the spectral fit is *larger* than $\sigma_{\dot P}$. For equilibrium accretors it is not; for propeller spin-down systems (AE Aqr, $\dot P \sim 6\times 10^{-14}$) and strong spin-up systems it is.
- It validates the joint-likelihood framework: it correctly identifies an observable as redundant rather than spuriously over-constraining.

Position in the paper: this becomes Section 5.1 (Bayesian parameter inference). Section 5.2 (spin evolution) follows naturally — the spin-equilibrium finding is the bridge.

---

## 2. Methodology — what is sound

- **Broad priors + observables, instead of using Salcedo's marginal $M$ posterior as a prior.** Right call. Using a spectral-derived $M$ prior would (i) double-count the $kT$ data, since $kT$ goes into the prior *and* the new likelihood, and (ii) discard the $M$-$\dot M$ covariance that lives in Salcedo's joint fit. Treating $L_X$ and $kT$ as independent likelihood terms with broad population priors recovers the joint constraint correctly through the physics.
- **Forward models**:
  - $\dot P$: Bhattacharyya & Chakrabarty (2017, Eq. 9), accretion regime. Standard.
  - $kT$: Rankine-Hugoniot strong shock with $v_{\rm ff}^2 = 2GM(1/R - 1/r_m)$, $\gamma=5/3$. Standard.
  - $L_X$: bolometric $GM\dot M (1/R - 1/r_m)$ scaled by a band-correction factor $f_{\rm band}(kT) = e^{-E_{\rm lo}/kT} - e^{-E_{\rm hi}/kT}$. The band correction is essential and correct — without it, predicted $L_X$ is ~2× the observed 0.3-12 keV value at $kT \sim 16$ keV.
- **$\xi$ marginalization** over a log-uniform prior $[0.3, 1.0]$ (Ghosh & Lamb 1979 range). This is a real improvement over Salcedo's MCVSPEC fits with implicit fixed-$\xi$. Since $B \propto \xi^{-7/4}$, this is the dominant systematic on $B$, and the notebook's $B$ posterior properly reflects it.
- **Mass prior** $\mathcal{N}(0.83, 0.20)$ from Pala+2022's volume-limited CV sample: weakly informative, well-justified.
- **Distance uncertainty**: $\sigma_d / d = 2.8\%$ propagated as $2\sigma_d/d$ into $\sigma_{L_X}$. Correct.
- **Sampler**: dynesty `DynamicNestedSampler` with 2000 live points, multi-ellipsoidal bounding, `rwalk` proposal. Appropriate for 4-D posteriors with hard boundaries. Convergence is well-resolved ($\sigma_{\ln Z} \sim 0.05$).
- **Hard rejection of $r_m > r_{\rm co}$** via NaN return + $-10^{100}$ likelihood: numerically stable.

---

## 3. Methodology — subtleties to flag

**Degrees of freedom.** IP has 4 parameters and 3 observables (formally $-1$ dof, under-constrained by data alone). The shortfall is filled by (a) the population mass prior and (b) the inequality constraint $R_{\rm WD} \le r_m \le r_{\rm co}$. This is fine, but worth acknowledging: the IP posterior in the $(\mu, \xi)$ plane is shaped by prior + boundary as much as by likelihood, and the resulting $B$ posterior is correspondingly broad (factor ~4 1$\sigma$ range). The width is honestly representing limited information, not a sampler problem.

**$kT$ observable choice (intentional simplification).** $kT_{\rm obs} = 16.3$ keV is from Salcedo's MKCFLOW fit (Table 6), which is the nmCV/boundary-layer cooling-flow model. The notebook uses this observable in the IP forward model. Justification: $kT_{\max}$ represents the high-energy continuum cutoff of bremsstrahlung emission, which is approximately model-independent. Salcedo's MCVSPEC IP fits (Table 7) don't tabulate $T_s$ directly but the implied range is consistent with ~15-25 keV. Acceptable simplification — flag in writing as a caveat. A more rigorous treatment would refit Salcedo's data with a unified spectral model, but that is out of scope for a methods demonstration.

**Propeller treatment** (hard exclusion via $r_m \le r_{\rm co}$).
- Past failure mode (the user's experience): if the kT/$L_X$ likelihood terms are dropped to admit propeller (since propeller has no shock-heated column), the sampler oversamples propeller because the $\dot P = 0$ likelihood by itself doesn't penalize it. This motivated the hard-rejection treatment.
- Why exclusion is justified physically: CJ2056 emits hard X-rays at $kT \sim 16$ keV with $L_X \sim 10^{31}$ erg/s and shows hardness-ratio modulation diagnostic of an accretion column (Salcedo+2024 §6.1-6.2). These are accretor signatures, not propeller. AE Aqr (the prototype propeller) has very different X-ray properties.
- In the paper: justify propeller exclusion in **one sentence** citing Salcedo's column-accretion evidence; do not introduce a third competing model. Including propeller as a parallel inference would either require dropping $kT/L_X$ from the likelihood (oversampling problem) or inventing ad-hoc kT predictions for propeller (unmotivated).

**$r_m / r_{\rm co}$ posterior breadth.** Median 0.70 with 68% CI $\sim [0.5, 0.9]$. The posterior covers both sub-equilibrium accretion ($x \sim 0.6$) and near-equilibrium configurations ($x \sim 0.9$); the data do not distinguish them ($1\sigma$ width $\sim 0.2$, so the median is within $\sim 1\sigma$ of any plausible equilibrium fastness). The right edge at $r_m / r_{\rm co} = 1$ is the propeller-prior boundary, not a peak. Don't overstate this — the right framing is *"consistent with near spin equilibrium"*, not *"at spin equilibrium"*.

---

## 4. The $\dot P$ non-leverage finding (the key result)

**User's hypothesis** (worth quoting): a tight $\dot P$ would constrain $B$ because the IP magnetic-torque term $\tau_{\rm mag} \propto \mu^2 / r_m^3$ depends explicitly on $\mu$ and hence $B$.

**Empirical result.** Comparing posteriors with and without the $\dot P$ likelihood term (cell 27 of the notebook):

| parameter | with $\dot P$ | spectral only |
|---|---|---|
| IP $M_{\rm WD}$ ($M_\odot$) | $0.871^{+0.129}_{-0.081}$ | $0.872^{+0.132}_{-0.083}$ |
| IP $\log_{10}\dot M$ | $14.723^{+0.033}_{-0.034}$ | $14.724^{+0.034}_{-0.035}$ |
| IP $\log_{10}\mu$ | $30.07^{+0.36}_{-0.33}$ | $30.11^{+0.34}_{-0.33}$ |
| IP $\xi$ | $0.57^{+0.27}_{-0.20}$ | $0.54^{+0.28}_{-0.18}$ |
| IP $\log_{10}(B/{\rm G})$ | $3.74^{+0.31}_{-0.29}$ | $3.79^{+0.30}_{-0.32}$ |

All parameters within $0.1\sigma$.

**Why** (mechanistic):
1. Spectral observables already pin $(M, \dot M)$ tightly via the $kT$-$M$ relation and the $L_X$-$\dot M$ relation.
2. The remaining freedom is in $(\mu, \xi)$. The implied $r_m / r_{\rm co}$ posterior is broad (1$\sigma$ width $\sim 0.2$) but lies within the accreting regime.
3. Across that posterior, predicted $|\dot P|$ stays within $\pm \sigma_{\dot P}$ for the bulk of the volume.
4. With $\sigma_{\dot P} = 1.7\times 10^{-15}$ s/s, observed $\dot P = 0$ is within ~1$\sigma$ for essentially all of the spectrally-allowed parameter space — so the $\dot P$ likelihood term cannot down-weight any region.

**Posterior-predictive check on $\dot P$ from the spectral-only fit.** Implemented by Cell 30 of the notebook (the `frac()` helper) and saved as `paper_figures/bayesian_predictive_pdot_specOnly.pdf`. The corner overlay shows the *symptom* (with-$\dot P$ vs spec-only posteriors are identical); pushing the spectral-only IP samples through `pdot_ip(...)` shows the *cause*. If predicted $|\dot P|$ is mostly within $\sigma_{\dot P}$ of zero, this is direct, mechanistic evidence that the spectral data already encode '$\dot P \approx 0$' — so adding the $\dot P = 0$ likelihood term cannot down-weight anything. This is the cleanest demonstration of the leverage finding.

**Physical interpretation.** CJ2056's spectral data place it in a sub-corotation accreting state consistent with near spin equilibrium. Across the spectrally-allowed posterior, predicted $|\dot P|$ stays within $\sigma_{\dot P}$, so $\dot P \approx 0$ is consistent with the entire allowed region — regardless of whether the system is exactly at equilibrium or somewhat sub-equilibrium.

**Generalization** (one of the section's takeaways):
- Tight $\dot P$ measurements add parameter leverage in CVs *away* from spin equilibrium — propeller spin-down systems ($|\dot P| \sim 10^{-13}$-$10^{-11}$), strong spin-up systems where $\dot P \gg \sigma_{\dot P}$.
- For equilibrium accretors, **spectral observables dominate**. Pulsar-timing $\dot P$ is best framed as a *consistency check on the equilibrium hypothesis* rather than a parameter constraint. This is a useful general lesson for the CV-timing community.

---

## 5. The $B$ discrepancy with Salcedo+2024 (treat as a finding)

- Notebook: $\log_{10}(B/{\rm G}) = 3.74^{+0.31}_{-0.29}$ → $B \approx 5.5$ kG.
- Salcedo+2024 (§6.3): $B \sim 0.1$-1 MG, citing Vermette+2023 + Norton+2004.

**Sources of the discrepancy**:
1. *Magnetospheric prefactor.* The notebook uses Ghosh & Lamb (1979), $r_m = \xi r_{\rm A}$, with $\xi$ marginalized over $[0.3, 1.0]$. Salcedo's pipeline has a different implicit prefactor pinning $r_m$ via Norton+2004. Since $B \propto \xi^{-7/4}$, a factor 2 in $\xi$ → factor 3.4 in $B$.
2. *Spin-equilibrium assumption.* Salcedo enforces $r_m = r_{\rm co}$ exactly. The notebook lets $r_m$ float in $[R, r_{\rm co}]$. The data prefer $r_m$ close to but not at $r_{\rm co}$, corresponding to slightly smaller $\mu$ (and $B$) than the equilibrium-locked case.
3. *$\dot M$ prescription.* Salcedo tabulates cases by fractional accretion area $f$; the notebook samples $\dot M$ as a free parameter constrained by $L_X$.

**Recommended framing**: present the notebook's $B$ as the more honest answer because it propagates (a) the $\xi$ prefactor systematic and (b) the joint $(M, \dot M)$ covariance. Salcedo's range overlaps the upper tail of the posterior at the small-$\xi$ end, so this is *not* a contradiction — it's a quantification of a systematic Salcedo's analysis didn't propagate.

Draft sentence for the paper: *"Marginalizing over the magnetospheric-radius prefactor $\xi$ and the joint $(M, \dot M)$ covariance, we obtain $\log_{10}(B/{\rm G}) = 3.74^{+0.31}_{-0.29}$, consistent with the upper tail of the previous estimate $B \sim 0.1$-1 MG (Salcedo et al. 2024) once their fixed-$\xi$, $r_m = r_{\rm co}$ assumption is relaxed."*

---

## 6. Recommended content for the paper section

**Include**:
- The unified-likelihood framework: priors, forward models, observables — short, mostly equations.
- IP posterior corner ($M$, $\log\dot M$, $\log B$) with priors overlaid: `bayesian_corner_ip_spectral.pdf`.
- $\dot P$ leverage corner overlay (with-$\dot P$ vs spectral-only): `bayesian_corner_ip_pdot_leverage.pdf`. **Key figure** for the section.
- Posterior-predictive $\dot P$ histogram from the spectral-only fit, with $\pm\sigma_{\dot P}$ band: `bayesian_predictive_pdot_specOnly.pdf` (already produced by Cell 30).
- Numerical leverage table (medians and 1$\sigma$ for the with-$\dot P$ vs spectral-only IP posteriors) — small in-text comparison, not a formal table necessarily.
- One paragraph on $r_m / r_{\rm co}$ as a spin-equilibrium diagnostic.
- One paragraph (one or two sentences, really) justifying propeller exclusion via Salcedo's column-accretion evidence.

**Exclude (or minimize)**:
- Pdot histogram from the with-$\dot P$ fit (covered by the spec-only predictive check).
- Detailed propeller comparison or third model.

**Suggested subsection layout**:
- 5.1.1 Bayesian framework (priors, forward models, likelihood).
- 5.1.2 Posterior parameter inference (M, $\dot M$, $B$ for the IP model, with priors overlaid).
- 5.1.3 The $\dot P$ leverage and spin equilibrium (the key scientific result, the methodological lesson, the predictive check).

---

## 7. Things to keep in mind while writing

1. **Don't over-commit to spin equilibrium.** The $r_m / r_{\rm co}$ posterior covers $\sim [0.5, 0.9]$ — broad enough that we cannot distinguish sub-equilibrium from near-equilibrium accretion. Phrase as *"consistent with near spin equilibrium"*, never *"at spin equilibrium"* or *"we find spin equilibrium"*. The honest equivalent claim that *is* defensible: across the spectrally-allowed posterior, predicted $|\dot P|$ is below $\sigma_{\dot P}$ — that's the load-bearing finding.

2. **Justify dynesty as a posterior sampler, not as an evidence calculator.** The motivating features here are (a) clean handling of the hard `r_m ≤ r_co` prior boundary via NaN-rejection in the likelihood, and (b) automatic convergence diagnostics with no burn-in tuning. We do not report Bayes factors, so do not frame nested sampling as a model-comparison tool — it just happens to be a good posterior sampler for this geometry.

3. **Frame the null result actively, not defensively.** Write "we test whether $\dot P$ adds parameter leverage and find it does not, as expected for a system near spin equilibrium" — not "unfortunately, $\dot P$ doesn't help." The negative result *is* the finding.

4. **Don't overclaim methodology novelty.** Bhattacharyya & Chakrabarty 2017 ($\dot P$ model), Vermette+2023 (MCVSPEC + $\xi$), Pala+2022 (mass prior), Ghosh & Lamb 1979 ($\xi$) all exist. The contribution is *combining* them in a single nested-sampling inference and demonstrating the $\dot P$-leverage diagnostic on a real source.

5. **Distinguish inputs from outputs in the prose.** Inputs: priors, hard $r_m$ bound, forward models, $L_X / kT / \dot P$ values. Outputs: posteriors, $B$. Don't blur these.

6. **Connect back to Section 4.** The PINT analysis spent considerable effort obtaining the tight $\dot P$. Section 5's "it doesn't add parameter leverage here" needs to be framed as still useful: (a) it's a consistency check on the equilibrium hypothesis, (b) it constrains long-term spin evolution (Section 5.2), (c) the methodology generalizes to systems where $\dot P$ does help.

7. **Section 5.2 (spin evolution) follows naturally from the equilibrium finding.** Posterior $r_m \approx r_{\rm co}$ → predicted $|\dot P|$ is small → equilibrium timescale $\tau = P/|\dot P|$ is much longer than the measurement floor → consistent with Salcedo's $\tau > 5\times 10^8$ yr. So 5.2 reuses the same posteriors.

8. **Citations to keep handy**:
   - Bhattacharyya & Chakrabarty 2017 ($\dot P$ model)
   - Vermette et al. 2023 (MCVSPEC + $\xi$ handling)
   - Pala et al. 2022 (CV mass distribution prior)
   - Ghosh & Lamb 1979 (magnetospheric prefactor $\xi$)
   - Norton et al. 2004 (alternative $r_m$ prescription, for the B comparison)
   - Salcedo et al. 2024 ($kT$, $L_X$, $B$ comparison, IP classification, accretion-column evidence)
   - Speagle 2020 (dynesty)
   - Rosen et al. 1988 (accretion curtain — supports propeller exclusion)